In [1]:
import pandas as pd 

df = pd.read_csv('umdhusky-data_collection-syscall-auto-gps-no-cam-processed.csv')


print(df.columns)

print(df.dtypes)

df.head()



Index(['Epoch Time', 'Hours', 'Minutes', 'Seconds', 'Milliseconds', 'Node',
       'PID', 'System Call', 'Unfinished Call', 'Resumed Call',
       'Execution Time', 'Return Code', 'Arguments'],
      dtype='object')
Epoch Time         float64
Hours                int64
Minutes              int64
Seconds              int64
Milliseconds         int64
Node                object
PID                  int64
System Call         object
Unfinished Call      int64
Resumed Call         int64
Execution Time     float64
Return Code        float64
Arguments           object
dtype: object


,Epoch Time,Hours,Minutes,Seconds,Milliseconds,Node,PID,System Call,Unfinished Call,Resumed Call,Execution Time,Return Code,Arguments
0,1.719506e+09,16,41,32,294,point_cloud_pipeline/point_cloud_pipeline,4395,futex,0,0,0.000086,1.0,"FUTEX_WAKE_PRIVATE, 1)"
1,1.719506e+09,16,41,32,294,point_cloud_pipeline/point_cloud_pipeline,4395,futex,1,0,NaN,NaN,"FUTEX_WAIT_BITSET_PRIVATE, 0, {tv_sec, tv_nsec..."
2,1.719506e+09,16,41,32,294,goto_object_behavior,4363,clock_nanosleep,0,1,0.050244,0.0,)
3,1.719506e+09,16,41,32,294,goto_object_behavior,4363,clock_nanosleep,1,0,NaN,NaN,"CLOCK_REALTIME, 0, {tv_sec, tv_nsec},"
4,1.719506e+09,16,41,32,294,front_ouster/ouster_driver,4248,futex,0,0,0.000011,1.0,"FUTEX_WAKE_PRIVATE, 1)"


In [14]:
unique_nodes = df['Node'].dropna().unique()
print(unique_nodes)

['point_cloud_pipeline/point_cloud_pipeline' 'goto_object_behavior'
 'front_ouster/ouster_driver' 'front_ouster/os_cloud_node'
 'local_planning_abstraction' 'global_planning_abstraction'
 'global_planner' 'omnigraph' 'visual_cache/visual_cache'
 'min_z_classifier' 'ioc_costmap_generator']


In [15]:
unique_syscalls_per_node = (
    df.groupby('Node')['System Call']
      .agg(lambda s: sorted(s.dropna().unique()))
      .reset_index(name='Unique System Calls')
)
unique_syscalls_per_node

,Node,Unique System Calls
0,front_ouster/os_cloud_node,"[clock_nanosleep, futex, write]"
1,front_ouster/ouster_driver,"[futex, recvfrom, select, write]"
2,global_planner,"[brk, clock_gettime, close, epoll_ctl, fstat, ..."
3,global_planning_abstraction,"[brk, futex, write]"
4,goto_object_behavior,[clock_nanosleep]
5,ioc_costmap_generator,[select]
6,local_planning_abstraction,"[futex, write]"
7,min_z_classifier,"[futex, write]"
8,omnigraph,"[brk, epoll_ctl, futex, sendto, times, write]"
9,point_cloud_pipeline/point_cloud_pipeline,"[brk, futex, write]"


In [16]:
unique_pairs = (
    df[['Node', 'System Call']]
      .dropna()
      .drop_duplicates()
      .sort_values(['Node', 'System Call'])
)
unique_pairs

,Node,System Call
34306,front_ouster/os_cloud_node,clock_nanosleep
6,front_ouster/os_cloud_node,futex
5,front_ouster/os_cloud_node,write
4,front_ouster/ouster_driver,futex
21,front_ouster/ouster_driver,recvfrom
20,front_ouster/ouster_driver,select
7,front_ouster/ouster_driver,write
798,global_planner,brk
785,global_planner,clock_gettime
724,global_planner,close


In [18]:
print(df['System Call'].unique())

['futex' 'clock_nanosleep' 'write' 'select' 'recvfrom' 'read' 'poll'
 'openat' 'close' 'clock_gettime' 'brk' 'times' 'sendto' 'epoll_ctl'
 'shutdown' 'fstat']


In [2]:
time_min = df['Epoch Time'].min()
time_max = df['Epoch Time'].max()
print(f"Epoch Time range: {time_min} to {time_max} (span: {time_max - time_min} seconds)")

Epoch Time range: 1719506492.2940538 to 1719506515.5650687 (span: 23.27101492881775 seconds)


In [2]:
# Check if there is any row where both 'Unfinished Call' and 'Resumed Call' are 1
has_both_unfinished_and_resumed = (
    ((df['Unfinished Call'] == 1) & (df['Resumed Call'] == 1)).any()
)

print(f"Is there any row with both Unfinished Call == 1 and Resumed Call == 1? {has_both_unfinished_and_resumed}")

# Optionally, show those rows:
rows_both = df[(df['Unfinished Call'] == 1) & (df['Resumed Call'] == 1)]
if not rows_both.empty:
    print("Rows with both Unfinished Call == 1 and Resumed Call == 1:")
    display(rows_both)

Is there any row with both Unfinished Call == 1 and Resumed Call == 1? False
